# InceptionResNetV2 Expérience A — Screening Complet Folds 3, 0 & 4

Ce notebook Google Colab permet d'exécuter l'entraînement, l'évaluation et la génération de rapports pour **InceptionResNetV2 Expérience A (High Performance)**.

## Protocole Scientifique Strict :
- **Architecture** : InceptionResNetV2 pré-entraîné sur ImageNet (`weights="imagenet"`)
- **Taille d'entrée** : $299 \times 299 \times 3$ en `float32` nativement dans $[0, 255]$
- **Preprocessing interne** : Couche sérialisable `tf.keras.layers.Rescaling(scale=1.0/127.5, offset=-1.0)` transformant $[0, 255] \to [-1, 1]$. Aucune division externe par 255.
- **Tête de classification** : Architecture `article_inspired` identique à DenseNet121 Exp D (GlobalAveragePooling2D → Dense 512 ELU → BatchNorm → Dropout 0.30 → Dense 128 ELU L2=0.01 → Dense 22 Softmax float32).
- **Manifeste autoritaire** : `data/manifests/densenet121_folds.csv` (432 images originales, 22 classes, seed 42).
- **Folds de Screening** : Folds **3, 0 et 4** dans cet ordre (259 images de validation au total : 86 + 87 + 86).
- **Augmentation** : Augmentation riche uniquement sur le train set ; images originales brutes pour la validation ; aucune TTA.
- **Stratégie 3-Phase Fine-Tuning** :
  - **Phase 1** : Entraînement de la tête seule (LR = 3e-4, backbone gelé).
  - **Phase 2** : Fine-tuning partiel (30% dernières couches, LR = 1e-5, BN frozen).
  - **Phase 3** : Fine-tuning complet (toutes les couches, LR = 3e-6, BN frozen).
- **Mixed precision** : `mixed_float16` avec sortie Softmax en `float32`.
- **⚠ Avertissement** : Ce système est à visée éducative uniquement. Il ne constitue en aucun cas un outil de diagnostic clinique.

## 1. Cellule Unique de Configuration Utilisateur

Tous les paramètres modifiables par l'utilisateur sont centralisés dans cette cellule.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURATION UTILISATEUR — InceptionResNetV2 Exp A High Performance
# ═══════════════════════════════════════════════════════════════════════════

REPO_URL = (
    "https://github.com/MyElhadri/"
    "histology-ai-classification.git"
)

BRANCH = "main"

PROJECT_DIR = "/content/histology-ai-classification"

DRIVE_DATASET = (
    "/content/drive/MyDrive/"
    "histology-ai-classification/"
    "data/nuinsseg_human_22_original"
)

LOCAL_DATASET = (
    "/content/histology-ai-classification/"
    "data/raw/nuinsseg_human_22_original"
)

OUTPUT_DIR = (
    "/content/drive/MyDrive/histology-results/"
    "inception-resnet-v2-exp-a-screening-folds-3-0-4"
)

CONFIG_PATH = (
    "configs/experiments/"
    "inception_resnet_v2_exp_a_high_performance.yaml"
)

FOLDS_TO_RUN = [3, 0, 4]

RUN_TESTS = True
RUN_DRY_RUN = True
RUN_TRAINING = True
GENERATE_REPORT = True
RUN_DENSENET_COMPARISON = True

SKIP_COMPLETED_FOLDS = True
BACKUP_PARTIAL_FOLDS = True
ALLOW_OVERWRITE = False

# Chemin configurable vers les prédictions DenseNet121 Exp D
DENSENET_OOF_DIR = (
    "/content/drive/MyDrive/histology-results/"
    "densenet121-exp-d/reports/densenet121/predictions"
)

print("Configuration Utilisateur InceptionResNetV2 chargée avec succès.")
print(f"Folds de screening sélectionnés : {FOLDS_TO_RUN}")
print(f"Dossier de sauvegarde Drive : {OUTPUT_DIR}")

## 2. Vérification du GPU et de l'Environnement TensorFlow

In [ ]:
import sys
import tensorflow as tf

print(f"Version de Python : {sys.version}")
print(f"Version de TensorFlow : {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError("AUCUN GPU DÉTECTÉ ! Veuillez activer un accélérateur GPU dans Colab (Exécution > Modifier le type d'exécution).")

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU Détecté et configuré (Memory Growth ON) : {gpu}")

# Mixed precision
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print(f"Mixed precision policy : {tf.keras.mixed_precision.global_policy().name}")

!nvidia-smi

## 3. Montage de Google Drive

In [ ]:
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

output_path = Path(OUTPUT_DIR)
output_path.mkdir(parents=True, exist_ok=True)
print(f"Google Drive monté. Dossier de sortie prêt : {output_path}")

## 4. Clonage ou Synchronisation du Dépôt GitHub

In [ ]:
import os
import subprocess

if not os.path.exists(PROJECT_DIR):
    print(f"Clonage du dépôt depuis {REPO_URL} (branche {BRANCH})...")
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, PROJECT_DIR], check=True)
else:
    print(f"Mise à jour du dépôt existant dans {PROJECT_DIR}...")
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=PROJECT_DIR, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=PROJECT_DIR, check=True)
    subprocess.run(["git", "pull", "origin", BRANCH], cwd=PROJECT_DIR, check=True)

os.chdir(PROJECT_DIR)
print(f"Dossier de travail actif : {os.getcwd()}")
!git status -s
!git log -1 --oneline

## 5. Vérification des Fichiers Obligatoires

In [ ]:
from pathlib import Path

required_files = [
    PROJECT_DIR + "/src/models/inception_resnet_v2.py",
    PROJECT_DIR + "/src/models/heads.py",
    PROJECT_DIR + "/scripts/train_inception_resnet_v2.py",
    PROJECT_DIR + "/scripts/generate_inception_resnet_v2_screening_report.py",
    PROJECT_DIR + "/scripts/compare_densenet_inception_resnet_v2_oof.py",
    PROJECT_DIR + "/" + CONFIG_PATH,
    PROJECT_DIR + "/data/manifests/densenet121_folds.csv",
    PROJECT_DIR + "/data/manifests/class_mapping.json",
]

missing_files = [f for f in required_files if not Path(f).is_file()]
if missing_files:
    raise FileNotFoundError(f"Fichiers requis manquants : {missing_files}")

print("Tous les fichiers obligatoires du projet sont présents et valides.")

## 6. Installation des Dépendances et Contrôle de Version

In [ ]:
import subprocess
import sys

req_colab = Path(PROJECT_DIR) / "requirements-colab.txt"
req_std = Path(PROJECT_DIR) / "requirements.txt"

target_req = req_colab if req_colab.is_file() else req_std
print(f"Installation des dépendances depuis {target_req}...")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(target_req)], check=True)

import numpy as np
import pandas as pd
import sklearn
import yaml

print("\n--- Versions des bibliothèques clés ---")
print(f"NumPy : {np.__version__}")
print(f"Pandas : {pd.__version__}")
print(f"Scikit-Learn : {sklearn.__version__}")
print(f"PyYAML : {yaml.__version__}")

## 7. Préparation et Vérification du Dataset d'Images Originales

In [ ]:
import os
from pathlib import Path

drive_ds = Path(DRIVE_DATASET)
local_ds = Path(LOCAL_DATASET)

if not drive_ds.is_dir():
    raise FileNotFoundError(f"Le dataset source Google Drive est introuvable à l'emplacement : {drive_ds}")

local_ds.parent.mkdir(parents=True, exist_ok=True)
if local_ds.is_symlink() or local_ds.exists():
    if local_ds.is_symlink():
        local_ds.unlink()

os.symlink(drive_ds, local_ds)
print(f"Lien symbolique créé : {local_ds} -> {drive_ds}")

valid_exts = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
image_files = [p for p in local_ds.rglob("*") if p.suffix.lower() in valid_exts]
print(f"Nombre total d'images trouvées dans le dataset : {len(image_files)}")

if len(image_files) != 432:
    raise ValueError(f"Le dataset doit contenir exactement 432 images originales, trouvé : {len(image_files)}")

print("Dataset vérifié avec succès (exactement 432 images originales).")

## 8. Vérification du Manifeste Autoritaire et de la Distribution des Folds

In [ ]:
import pandas as pd
from pathlib import Path

manifest_path = Path(PROJECT_DIR) / "data/manifests/densenet121_folds.csv"
df_manifest = pd.read_csv(manifest_path)

print(f"Total d'images dans le manifeste autoritaire : {len(df_manifest)}")
assert len(df_manifest) == 432, "Le manifeste doit contenir 432 lignes."
assert df_manifest["image_id"].nunique() == 432, "Tous les image_id doivent être uniques."

fold_counts = df_manifest["fold"].value_counts().sort_index().to_dict()
print("\nRépartition exacte des images par fold :")
for f_id, count in fold_counts.items():
    print(f" - Fold {f_id} : {count} images")

expected_counts = {0: 87, 1: 87, 2: 86, 3: 86, 4: 86}
for f_id, exp_c in expected_counts.items():
    assert fold_counts[f_id] == exp_c, f"Erreur sur le fold {f_id} : attendu {exp_c}, obtenu {fold_counts[f_id]}"

# Vérifier que les folds de screening totalisent 259
screening_total = sum(fold_counts[f] for f in FOLDS_TO_RUN)
assert screening_total == 259, f"Total screening attendu 259, obtenu {screening_total}"

print(f"\nValidation réussie : 432 images totales, screening folds {FOLDS_TO_RUN} = {screening_total} images.")

## 9. Inspection et Validation de la Configuration InceptionResNetV2

In [ ]:
import yaml
from pathlib import Path

yaml_file = Path(PROJECT_DIR) / CONFIG_PATH
with open(yaml_file, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

print("=== CONFIGURATION INCEPTIONRESNETV2 EXP A ===")
print(f"Architecture : {cfg['model']['architecture']}")
print(f"Taille d'entrée : {cfg['model']['input_shape']}")
print(f"Nombre de classes : {cfg['model']['num_classes']}")
print(f"Poids d'origine : {cfg['model']['weights']}")
print(f"Tête de classification : {cfg['model']['classifier_head']['type']}")
print(f"Preprocessing méthode : {cfg['preprocessing']['method']}")
print(f"Folds de screening : {cfg['screening']['folds']}")
print(f"Mixed precision : {cfg['training']['mixed_precision']}")
print(f"Phases d'entraînement : 3 (Phase 1: {cfg['training']['phase_1']['epochs']}ep, "
      f"Phase 2: {cfg['training']['phase_2']['epochs']}ep, "
      f"Phase 3: {cfg['training']['phase_3']['epochs']}ep)")

assert cfg['model']['architecture'] == "InceptionResNetV2"
assert cfg['model']['input_shape'] == [299, 299, 3]
assert cfg['model']['num_classes'] == 22
assert cfg['preprocessing']['external_divide_by_255'] is False
assert cfg['preprocessing']['external_preprocess_input'] is False
assert cfg['screening']['folds'] == [3, 0, 4]
assert cfg['training']['mixed_precision'] is True
assert cfg['model']['classifier_head']['output_activation'] == 'softmax'
print("\nValidation automatique de la configuration YAML réussie.")

## 10. Exécution des Tests Unitaires Automatisés (Pytest)

In [ ]:
import subprocess
import sys

if RUN_TESTS:
    print("Lancement de la suite de tests unitaires ciblés InceptionResNetV2...\n")
    test_cmd = [
        sys.executable, "-m", "pytest",
        "tests/test_inception_resnet_v2.py",
        "tests/test_inception_resnet_v2_report.py",
        "tests/test_compare_densenet_inception_resnet_v2_oof.py",
        "-v"
    ]
    res = subprocess.run(test_cmd, cwd=PROJECT_DIR)
    if res.returncode != 0:
        raise RuntimeError("ÉCHEC DES TESTS UNITAIRES ! Corrigez les erreurs avant d'entraîner le modèle.")
    print("\nTous les tests unitaires ont réussi avec succès.")
else:
    print("RUN_TESTS est défini sur False. Étape de test ignorée.")

## 11. Dry-Run : Validation sans Entraînement

In [ ]:
import os
import subprocess
import sys

if RUN_DRY_RUN:
    print("Exécution du dry-run (validation config/modèle, weights=None, pas de model.fit)...\n")
    dryrun_cmd = [
        sys.executable, "-u",
        "scripts/train_inception_resnet_v2.py",
        "--config", CONFIG_PATH,
        "--dataset-dir", LOCAL_DATASET,
        "--output-dir", OUTPUT_DIR,
        "--folds", "3",
        "--dry-run"
    ]
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONPATH"] = PROJECT_DIR + os.pathsep + env.get("PYTHONPATH", "")

    proc = subprocess.Popen(
        dryrun_cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        cwd=PROJECT_DIR,
        env=env
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Dry-run ÉCHOUÉ (code {proc.returncode})")
    print("\nDry-run réussi : configuration et modèle validés sans entraînement.")
else:
    print("RUN_DRY_RUN est défini sur False. Dry-run ignoré.")

## 12. État des Résultats Existants et Reprise après Interruption

In [ ]:
import gc
import json
import os
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

ARCH_NAME = "inception_resnet_v2"
out_dir = Path(OUTPUT_DIR)
status_rows = []

for f in [0, 1, 2, 3, 4]:
    ckpt = out_dir / "models" / ARCH_NAME / "checkpoints" / f"fold_{f}" / "best_model.keras"
    met = out_dir / "reports" / ARCH_NAME / "metrics" / f"fold_{f}.json"
    oof = out_dir / "reports" / ARCH_NAME / "predictions" / f"fold_{f}_oof_predictions.csv"
    h_json = out_dir / "reports" / ARCH_NAME / "history" / f"fold_{f}_history.json"
    h_csv = out_dir / "reports" / ARCH_NAME / "history" / f"fold_{f}_history.csv"
    cw_json = out_dir / "reports" / ARCH_NAME / "class_weights" / f"fold_{f}_class_weights.json"
    comp_json = out_dir / "reports" / ARCH_NAME / "completion" / f"fold_{f}_complete.json"

    # Check completion marker
    is_complete = False
    if comp_json.is_file() and comp_json.stat().st_size > 0:
        try:
            with open(comp_json, "r", encoding="utf-8") as fp:
                marker = json.load(fp)
            is_complete = marker.get("complete", False)
        except Exception:
            pass

    if not is_complete:
        is_complete = all(
            x.is_file() and x.stat().st_size > 0
            for x in [ckpt, met, oof, h_json, h_csv, cw_json]
        )

    has_partial = any(
        x.is_file() for x in [ckpt, met, oof, h_json]
    ) and not is_complete

    status_rows.append({
        "fold": f,
        "complete": is_complete,
        "partial": has_partial,
        "checkpoint": ckpt.is_file(),
        "metrics": met.is_file(),
        "oof_predictions": oof.is_file(),
        "history": h_json.is_file(),
        "class_weights": cw_json.is_file(),
    })

df_status = pd.DataFrame(status_rows)
print("=== ÉTAT DES OUTPUTS PAR FOLD SUR GOOGLE DRIVE ===")
print(df_status.to_string(index=False))

## 13. Smoke Test de l'Architecture InceptionResNetV2

In [ ]:
import gc
import numpy as np
import tensorflow as tf
from src.models.inception_resnet_v2 import build_inception_resnet_v2, verify_no_double_preprocessing

print("Instanciation d'un modèle InceptionResNetV2 sans poids (weights=None) pour valider les formes...")
smoke_model = build_inception_resnet_v2(
    num_classes=22,
    input_shape=(299, 299, 3),
    weights=None,
    head_config=cfg['model']['classifier_head']
)

assert smoke_model.input_shape == (None, 299, 299, 3)
assert smoke_model.output_shape == (None, 22)
verify_no_double_preprocessing(smoke_model)

dummy_batch = np.random.uniform(0.0, 255.0, size=(2, 299, 299, 3)).astype(np.float32)
probs = smoke_model.predict(dummy_batch, verbose=0)
assert probs.shape == (2, 22)
assert probs.dtype == np.float32, f"Output dtype must be float32, got {probs.dtype}"
np.testing.assert_allclose(np.sum(probs, axis=1), [1.0, 1.0], rtol=1e-4)

print(f"Smoke test réussi : Formes (2, 299, 299, 3) -> (2, 22), Softmax float32 validés.")

del smoke_model
tf.keras.backend.clear_session()
gc.collect()

# 🚀 LANCEMENT DU SCREENING INCEPTIONRESNETV2 — FOLDS 3, 0 ET 4

Chaque fold est lancé séparément via `subprocess.Popen` avec logs en temps réel.
- Les folds déjà complets sont automatiquement ignorés (`SKIP_COMPLETED_FOLDS=True`)
- Les folds partiels sont sauvegardés avec timestamp (`BACKUP_PARTIAL_FOLDS=True`)
- Une interruption du fold 4 ne relance PAS les folds 3 et 0

In [ ]:
import gc
import json
import os
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import tensorflow as tf

ARCH_NAME = "inception_resnet_v2"


def is_fold_complete(out_dir, fold):
    """Check if a fold has all required outputs and valid completion marker."""
    od = Path(out_dir)
    ckpt = od / "models" / ARCH_NAME / "checkpoints" / f"fold_{fold}" / "best_model.keras"
    met = od / "reports" / ARCH_NAME / "metrics" / f"fold_{fold}.json"
    oof = od / "reports" / ARCH_NAME / "predictions" / f"fold_{fold}_oof_predictions.csv"
    h_json = od / "reports" / ARCH_NAME / "history" / f"fold_{fold}_history.json"
    h_csv = od / "reports" / ARCH_NAME / "history" / f"fold_{fold}_history.csv"
    cw = od / "reports" / ARCH_NAME / "class_weights" / f"fold_{fold}_class_weights.json"
    comp = od / "reports" / ARCH_NAME / "completion" / f"fold_{fold}_complete.json"

    required = [ckpt, met, oof, h_json, h_csv, cw, comp]
    if not all(f.is_file() and f.stat().st_size > 0 for f in required):
        return False

    try:
        with open(comp, "r", encoding="utf-8") as fp:
            marker = json.load(fp)
        if not marker.get("complete", False):
            return False
    except Exception:
        return False

    # Validate OOF row count
    import pandas as pd
    expected_sizes = {0: 87, 1: 87, 2: 86, 3: 86, 4: 86}
    try:
        oof_df = pd.read_csv(oof)
        if len(oof_df) != expected_sizes.get(fold, 0):
            return False
    except Exception:
        return False

    return True


def is_fold_partial(out_dir, fold):
    """Check if fold has partial outputs."""
    od = Path(out_dir)
    check_files = [
        od / "models" / ARCH_NAME / "checkpoints" / f"fold_{fold}" / "best_model.keras",
        od / "reports" / ARCH_NAME / "metrics" / f"fold_{fold}.json",
        od / "reports" / ARCH_NAME / "predictions" / f"fold_{fold}_oof_predictions.csv",
        od / "reports" / ARCH_NAME / "history" / f"fold_{fold}_history.json",
    ]
    return any(f.is_file() for f in check_files) and not is_fold_complete(out_dir, fold)


def backup_partial_fold(out_dir, fold):
    """Rename partial fold checkpoint dir with timestamp."""
    od = Path(out_dir)
    ckpt_dir = od / "models" / ARCH_NAME / "checkpoints" / f"fold_{fold}"
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_dir = ckpt_dir.parent / f"fold_{fold}_interrupted_{ts}"
    if ckpt_dir.exists():
        ckpt_dir.rename(backup_dir)
        print(f"Partial fold {fold} backed up to: {backup_dir}")
    return backup_dir


def validate_completed_fold(out_dir, fold):
    """Validate a completed fold: load model and verify 22-class output."""
    od = Path(out_dir)
    ckpt = od / "models" / ARCH_NAME / "checkpoints" / f"fold_{fold}" / "best_model.keras"
    if not ckpt.is_file():
        return False
    try:
        m = tf.keras.models.load_model(str(ckpt), compile=False)
        assert m.output_shape == (None, 22)
        test_batch = np.random.uniform(0.0, 255.0, size=(1, 299, 299, 3)).astype(np.float32)
        p = m.predict(test_batch, verbose=0)
        assert p.shape == (1, 22)
        np.testing.assert_allclose(np.sum(p), 1.0, rtol=1e-3)
        del m
        tf.keras.backend.clear_session()
        gc.collect()
        return True
    except Exception as e:
        print(f"Fold {fold} validation failed: {e}")
        return False


if RUN_TRAINING:
    print(f"Exécution fold par fold des folds {FOLDS_TO_RUN} en temps réel unbuffered...\n")

    for fold in FOLDS_TO_RUN:
        print(f"\n{'='*60}")
        print(f">>> TRAITEMENT DU FOLD {fold}")
        print(f"{'='*60}\n")

        # Check fold completion
        if is_fold_complete(OUTPUT_DIR, fold) and not ALLOW_OVERWRITE:
            if SKIP_COMPLETED_FOLDS:
                if validate_completed_fold(OUTPUT_DIR, fold):
                    print(f"Fold {fold} : COMPLET et VALIDÉ. Ignoré (SKIP_COMPLETED_FOLDS=True).")
                    continue
                else:
                    print(f"Fold {fold} : marqué comme complet mais validation échouée. Relancement.")
            else:
                print(f"Fold {fold} : complet. Utilisez ALLOW_OVERWRITE=True pour relancer.")
                continue

        # Handle partial fold
        if is_fold_partial(OUTPUT_DIR, fold):
            print(f"Fold {fold} : résultats PARTIELS détectés.")
            if BACKUP_PARTIAL_FOLDS:
                backup_partial_fold(OUTPUT_DIR, fold)
            else:
                print(f"Fold {fold} : résultats partiels laissés en place.")

        # Launch training for this fold
        cmd_fold = [
            sys.executable, "-u", "scripts/train_inception_resnet_v2.py",
            "--config", CONFIG_PATH,
            "--dataset-dir", LOCAL_DATASET,
            "--output-dir", OUTPUT_DIR,
            "--folds", str(fold),
        ]
        if SKIP_COMPLETED_FOLDS:
            cmd_fold.append("--skip-completed")
        if BACKUP_PARTIAL_FOLDS:
            cmd_fold.append("--backup-partial")
        if ALLOW_OVERWRITE:
            cmd_fold.append("--overwrite")

        env = os.environ.copy()
        env["PYTHONUNBUFFERED"] = "1"
        env["PYTHONPATH"] = PROJECT_DIR + os.pathsep + env.get("PYTHONPATH", "")

        proc = subprocess.Popen(
            cmd_fold,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            cwd=PROJECT_DIR,
            env=env
        )

        for line in proc.stdout:
            print(line, end="", flush=True)

        proc.wait()
        if proc.returncode != 0:
            raise RuntimeError(f"ÉCHEC LORS DE L'ENTRAÎNEMENT DU FOLD {fold} (Code d'erreur : {proc.returncode})")

    print("\nLancement et exécution de l'entraînement des folds terminés avec succès.")
else:
    print("\nRUN_TRAINING est défini sur False. Entraînement ignoré.")

## 15. Validation et Contrôle des Checkpoints Générés

In [ ]:
import gc
import numpy as np
import tensorflow as tf
from pathlib import Path

out_dir = Path(OUTPUT_DIR)

for f in FOLDS_TO_RUN:
    ckpt_file = out_dir / "models" / ARCH_NAME / "checkpoints" / f"fold_{f}" / "best_model.keras"
    if not ckpt_file.is_file():
        print(f"AVERTISSEMENT : Checkpoint manquant pour le fold {f}")
        continue

    size_mb = ckpt_file.stat().st_size / (1024 * 1024)
    print(f"Validation du checkpoint Fold {f} ({size_mb:.2f} MB)...")

    m = tf.keras.models.load_model(str(ckpt_file), compile=False)
    assert m.output_shape == (None, 22)

    test_batch = np.random.uniform(0.0, 255.0, size=(1, 299, 299, 3)).astype(np.float32)
    p = m.predict(test_batch, verbose=0)
    assert p.shape == (1, 22)
    assert p.dtype == np.float32, f"Output must be float32, got {p.dtype}"
    np.testing.assert_allclose(np.sum(p), 1.0, rtol=1e-4)
    print(f" -> Checkpoint Fold {f} valide (Softmax Sum = {np.sum(p):.4f}, dtype={p.dtype})")

    del m
    tf.keras.backend.clear_session()
    gc.collect()

## 16. Analyse des Métriques du Screening (Folds 3, 0, 4)

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

out_dir = Path(OUTPUT_DIR)
metrics_list = []

for f in FOLDS_TO_RUN:
    m_file = out_dir / "reports" / ARCH_NAME / "metrics" / f"fold_{f}.json"
    if m_file.is_file():
        with open(m_file, "r", encoding="utf-8") as fp:
            d = json.load(fp)
            d["fold"] = f
            metrics_list.append(d)

if metrics_list:
    df_metrics = pd.DataFrame(metrics_list)
    print("=== TABLEAU DES MÉTRIQUES INDIVIDUELLES DU SCREENING ===")
    cols_show = ["fold", "accuracy", "macro_f1", "weighted_f1", "correct_samples", "total_samples", "training_duration_seconds"]
    print(df_metrics[[c for c in cols_show if c in df_metrics.columns]].to_string(index=False))

    print("\n=== SYNTHÈSE MOYENNE ± ÉCART-TYPE (SCREENING 259 IMAGES) ===")
    print(f"Accuracy Moyenne    : {df_metrics['accuracy'].mean():.4f} ± {df_metrics['accuracy'].std():.4f}")
    print(f"Macro F1 Moyen      : {df_metrics['macro_f1'].mean():.4f} ± {df_metrics['macro_f1'].std():.4f}")
    print(f"Weighted F1 Moyen   : {df_metrics['weighted_f1'].mean():.4f} ± {df_metrics['weighted_f1'].std():.4f}")
else:
    print("Aucun fichier de métriques disponible pour les folds exécutés.")

## 17. Consolidation des Prédictions Out-Of-Fold (OOF) du Screening

In [ ]:
import pandas as pd
from pathlib import Path

out_dir = Path(OUTPUT_DIR)
oof_files = [out_dir / "reports" / ARCH_NAME / "predictions" / f"fold_{f}_oof_predictions.csv" for f in FOLDS_TO_RUN]
existing_oof = [f for f in oof_files if f.is_file()]

if existing_oof:
    dfs = [pd.read_csv(f) for f in existing_oof]
    combined_oof = pd.concat(dfs, ignore_index=True)

    print(f"Prédictions OOF consolidées pour {len(combined_oof)} images uniques de validation.")
    assert len(combined_oof) == 259, f"Attendu 259 images pour screening folds 3, 0, 4, obtenu : {len(combined_oof)}"

    save_oof_path = out_dir / "reports" / ARCH_NAME / "predictions" / "inception_resnet_v2_screening_oof_predictions.csv"
    save_oof_path.parent.mkdir(parents=True, exist_ok=True)
    combined_oof.to_csv(save_oof_path, index=False)
    print(f"CSV OOF de screening sauvegardé à : {save_oof_path}")
else:
    print("Aucune prédiction OOF trouvée pour la consolidation.")

## 18. Génération des Matrices de Confusion et Rapports de Présentation

In [ ]:
import os
import subprocess
import sys

if GENERATE_REPORT:
    print("Exécution du script de génération des rapports (generate_inception_resnet_v2_screening_report.py)...")
    cmd_rep = [
        sys.executable, "scripts/generate_inception_resnet_v2_screening_report.py",
        "--output-dir", OUTPUT_DIR
    ]
    env_rep = os.environ.copy()
    env_rep["PYTHONPATH"] = PROJECT_DIR + os.pathsep + env_rep.get("PYTHONPATH", "")
    res = subprocess.run(cmd_rep, check=True, cwd=PROJECT_DIR, env=env_rep)
    print("\nRapports de présentation et matrices 300 DPI générés avec succès.")
else:
    print("GENERATE_REPORT est défini sur False. Génération de rapport ignorée.")

## 19. Visualisation des Courbes d'Entraînement par Fold

In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path

out_dir = Path(OUTPUT_DIR)
curves_dir = out_dir / "reports" / ARCH_NAME / "training_curves"
curves_dir.mkdir(parents=True, exist_ok=True)

for f in FOLDS_TO_RUN:
    hist_file = out_dir / "reports" / ARCH_NAME / "history" / f"fold_{f}_history.json"
    if not hist_file.is_file():
        continue
    with open(hist_file, "r", encoding="utf-8") as fp:
        h = json.load(fp)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=300)

    axes[0].plot(h.get("accuracy", []), label="Train Accuracy")
    axes[0].plot(h.get("val_accuracy", []), label="Val Accuracy")
    for pb in h.get("phase_boundaries", []):
        axes[0].axvline(x=pb, linestyle="--", color="gray", alpha=0.7)
    axes[0].set_title(f"InceptionResNetV2 Fold {f} — Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(h.get("loss", []), label="Train Loss")
    axes[1].plot(h.get("val_loss", []), label="Val Loss")
    for pb in h.get("phase_boundaries", []):
        axes[1].axvline(x=pb, linestyle="--", color="gray", alpha=0.7)
    axes[1].set_title(f"InceptionResNetV2 Fold {f} — Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(curves_dir / f"fold_{f}_curves.png", dpi=300)
    plt.close(fig)
    print(f"Courbes enregistrées pour le Fold {f} -> {curves_dir / f'fold_{f}_curves.png'}")

## 20. Comparaison avec DenseNet121 Exp D et Analyse de Complémentarité

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

print("=== RÉFÉRENCE DENSENET121 EXP D (SCREENING FOLDS 0, 3, 4) ===")
print(" - Mean Accuracy   : ~0.8610")
print(" - Mean Macro F1   : ~0.7889")
print(" - Mean Weighted F1: ~0.8538\n")

densenet_pred_dir = Path(DENSENET_OOF_DIR)
irv2_pred_dir = Path(OUTPUT_DIR) / "reports" / ARCH_NAME / "predictions"
comp_output_dir = Path(OUTPUT_DIR) / "reports" / ARCH_NAME / "densenet_comparison"

if RUN_DENSENET_COMPARISON and densenet_pred_dir.is_dir() and irv2_pred_dir.is_dir():
    # Check if DenseNet OOF prediction files exist for the screening folds
    dense_files_exist = all(
        (densenet_pred_dir / f"fold_{f}_oof_predictions.csv").is_file() for f in FOLDS_TO_RUN
    )
    if dense_files_exist:
        print("Exécution de l'analyse de complémentarité OOF...")
        cmd_comp = [
            sys.executable, "scripts/compare_densenet_inception_resnet_v2_oof.py",
            "--densenet-dir", str(densenet_pred_dir),
            "--inception-resnet-v2-dir", str(irv2_pred_dir),
            "--output-dir", str(comp_output_dir),
        ]
        env_comp = os.environ.copy()
        env_comp["PYTHONPATH"] = PROJECT_DIR + os.pathsep + env_comp.get("PYTHONPATH", "")
        res = subprocess.run(cmd_comp, capture_output=True, text=True, cwd=PROJECT_DIR, env=env_comp)
        print(res.stdout)
        if res.returncode != 0:
            print(f"Comparaison échouée (code {res.returncode}).")
            if res.stderr:
                print(f"Erreur : {res.stderr[:500]}")
        elif (comp_output_dir / "comparison_summary.json").is_file():
            with open(comp_output_dir / "comparison_summary.json", "r", encoding="utf-8") as fp:
                comp_res = json.load(fp)
            print("\n=== Synthèse de Complémentarité & Ensemble 50/50 ===")
            print("Taux de désaccord entre modèles :", comp_res["complementarity_breakdown"]["disagreement_rate"])
            print("Erreurs corrigées par InceptionResNetV2 :", comp_res["complementarity_breakdown"]["inception_resnet_v2_only_correct"])
            print("Erreurs corrigées par DenseNet :", comp_res["complementarity_breakdown"]["densenet_only_correct"])
            print("Accuracy Ensemble 50/50 :", comp_res["simple_ensemble_50_50"]["accuracy"])
    else:
        print(f"Prédictions DenseNet121 non trouvées dans {densenet_pred_dir}.")
        print("Comparaison ignorée. Le rapport InceptionResNetV2 reste valide.")
else:
    if not RUN_DENSENET_COMPARISON:
        print("RUN_DENSENET_COMPARISON est False. Comparaison ignorée.")
    else:
        print(f"Répertoire DenseNet OOF ({DENSENET_OOF_DIR}) introuvable.")
        print("Comparaison ignorée. Le rapport InceptionResNetV2 reste valide.")

## 21. Résumé Final et Verdict d'Exécution

In [ ]:
from pathlib import Path

out_dir = Path(OUTPUT_DIR)
screening_folds = [3, 0, 4]

checkpoints_valid = all(
    (out_dir / "models" / ARCH_NAME / "checkpoints" / f"fold_{f}" / "best_model.keras").is_file()
    for f in screening_folds
)
metrics_valid = all(
    (out_dir / "reports" / ARCH_NAME / "metrics" / f"fold_{f}.json").is_file()
    for f in screening_folds
)
oof_valid = all(
    (out_dir / "reports" / ARCH_NAME / "predictions" / f"fold_{f}_oof_predictions.csv").is_file()
    for f in screening_folds
)
summary_file = out_dir / "reports" / ARCH_NAME / "inception_resnet_v2_screening_summary.json"

if checkpoints_valid and metrics_valid and oof_valid and summary_file.is_file():
    print("\n" + "="*60)
    print("INCEPTIONRESNETV2 EXP A SCREENING COMPLETE")
    print("="*60 + "\n")
    print(f"Tous les artefacts de screening sont enregistrés dans Google Drive : {OUTPUT_DIR}")
else:
    print("\nCertains artefacts de screening sont manquants. Vérifiez les logs ci-dessus.")

## 22. Note sur les Folds 1 et 2

**Les folds 1 et 2 ne seront PAS lancés automatiquement.**

Si les critères de décision (Accuracy ≥ 0.84, Macro F1 ≥ 0.76 ou complémentarité d'ensemble positive) sont satisfaits après ce screening, vous pourrez entraîner les Folds 1 et 2 pour obtenir l'évaluation complète sur 432 images en modifiant la configuration initiale :

```python
FOLDS_TO_RUN = [1, 2]
```

Ne pas lancer automatiquement les folds 1 et 2 sans analyse du screening.